# Heart Disease Prediction — Exploratory Data Analysis

This notebook explores the UCI Heart Disease dataset (303 samples, 13 features, binary target).
We check data quality, visualise distributions, and derive insights before modelling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 20)

df = pd.read_csv('../data/heart.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Data Quality Check

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print(f'\n--- Missing Values ---')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')
print(f'\n--- Duplicates ---')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'\n--- Summary Statistics ---')
df.describe()

## 2. Target Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['target'].value_counts()
bars = ax.bar(['No Disease (0)', 'Disease (1)'], counts.values,
              color=['#3498db', '#e74c3c'])
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Count')
ax.set_title('Target Variable Distribution', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Class ratio — Disease: {counts[1]/len(df):.1%}, No Disease: {counts[0]/len(df):.1%}')

The dataset is roughly balanced (54% disease vs 46% no disease), so we don't need techniques like SMOTE. However, we still use stratified splitting to preserve this ratio in train/test sets.

## 3. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTop correlations with target:')
print(corr['target'].drop('target').sort_values(ascending=False))

## 4. Feature Distributions by Target

In [ ]:
key_features = ['age', 'chol', 'thalach', 'trestbps', 'oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, feat in enumerate(key_features):
    sns.histplot(data=df, x=feat, hue='target', kde=True,
                 palette={0: '#3498db', 1: '#e74c3c'},
                 ax=axes[i], stat='density', common_norm=False)
    axes[i].set_title(f'{feat} by Target', fontweight='bold')
axes[-1].axis('off')
plt.suptitle('Feature Distributions by Heart Disease Status', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Box Plots — Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for i, feat in enumerate(key_features):
    sns.boxplot(data=df, y=feat, x='target', palette='Set2', ax=axes[i])
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_xticklabels(['No Disease', 'Disease'])
plt.suptitle('Box Plots — Outlier Detection', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## Key EDA Insights

1. **No missing values** — all 303 rows × 14 columns are complete, so no imputation is needed.
2. **Roughly balanced classes** — 165 disease vs 138 no-disease (54%/46%), no resampling required.
3. **Strongest predictors** — `cp` (chest pain type), `thalach` (max heart rate), `exang` (exercise-induced angina), `oldpeak` (ST depression), and `slope` show the highest correlation with the target.
4. **Max heart rate (`thalach`)** is notably higher in disease-positive patients, while `oldpeak` is higher in disease-negative patients.
5. **Outliers exist** in `chol` (cholesterol > 400) and `trestbps` (resting BP > 170), but we keep them because they represent real clinical values and the dataset is small.